1. Setting up and loading the data

In [ ]:

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             silhouette_score, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# Connect to DuckDB warehouse
duck = duckdb.connect("../data/shopstream.duckdb")

# Load all tables
customers   = duck.execute("SELECT * FROM dim_customers").df()
products    = duck.execute("SELECT * FROM dim_products").df()
orders      = duck.execute("SELECT * FROM fct_orders").df()
rfm         = duck.execute("SELECT * FROM fct_customer_metrics").df()

print(f"Customers : {len(customers):,}")
print(f"Products  : {len(products):,}")
print(f"Orders    : {len(orders):,}")
print(f"RFM table : {len(rfm):,}")
rfm.head()

IOException: IO Error: Cannot open file "\\?\C:\Users\Navdeep\Desktop\VS CODE project\shopstream\notebooks\data\shopstream.duckdb": The system cannot find the path specified.


2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("ShopStream — Exploratory Data Analysis", fontsize=16, fontweight="bold")

# 1. Revenue by month
monthly = orders[orders["status"] == "completed"].groupby(
    ["order_year", "order_month"])["order_total"].sum().reset_index()
monthly["period"] = monthly["order_year"].astype(str) + "-" + \
                    monthly["order_month"].astype(str).str.zfill(2)
monthly = monthly.sort_values("period")
axes[0,0].plot(monthly["period"], monthly["order_total"], marker="o", linewidth=2)
axes[0,0].set_title("Monthly revenue (completed orders)")
axes[0,0].set_xlabel("Month")
axes[0,0].set_ylabel("Revenue ($)")
axes[0,0].tick_params(axis="x", rotation=45)

# 2. Orders by status
status_counts = orders["status"].value_counts()
axes[0,1].bar(status_counts.index, status_counts.values, color=sns.color_palette("husl", 4))
axes[0,1].set_title("Orders by status")
axes[0,1].set_ylabel("Count")

# 3. Revenue by product category
cat_rev = duck.execute("""
    SELECT p.category, ROUND(SUM(oi.line_total), 2) AS revenue
    FROM raw_order_items oi
    JOIN raw_products p ON oi.product_id = p.product_id
    JOIN raw_orders o   ON oi.order_id   = o.order_id
    WHERE o.status = 'completed'
    GROUP BY p.category ORDER BY revenue DESC
""").df()
axes[0,2].barh(cat_rev["category"], cat_rev["revenue"], color=sns.color_palette("husl", 5))
axes[0,2].set_title("Revenue by category")
axes[0,2].set_xlabel("Revenue ($)")

# 4. RFM segment distribution
seg_counts = rfm["rfm_segment"].value_counts()
axes[1,0].pie(seg_counts.values, labels=seg_counts.index, autopct="%1.1f%%",
              colors=sns.color_palette("husl", 4))
axes[1,0].set_title("RFM customer segments")

# 5. Monetary distribution
axes[1,1].hist(rfm["monetary"].dropna(), bins=40, edgecolor="white", linewidth=0.5)
axes[1,1].set_title("Customer lifetime spend distribution")
axes[1,1].set_xlabel("Total spend ($)")
axes[1,1].set_ylabel("Customers")

# 6. Recency vs Monetary
scatter = axes[1,2].scatter(rfm["recency_days"], rfm["monetary"],
                             c=rfm["rfm_score"], cmap="RdYlGn",
                             alpha=0.6, s=30)
plt.colorbar(scatter, ax=axes[1,2], label="RFM score")
axes[1,2].set_title("Recency vs monetary (color = RFM score)")
axes[1,2].set_xlabel("Recency (days)")
axes[1,2].set_ylabel("Total spend ($)")

plt.tight_layout()
plt.savefig("notebooks/eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("EDA complete!")

3. Churn Prediction using Random Forest

In [ ]:
# ── Build churn label ────────────────────────────────────────
# A customer is "churned" if they haven't ordered in 180+ days
churn_df = rfm[rfm["rfm_segment"].notna()].copy()
churn_df["churned"] = (churn_df["recency_days"] >= 180).astype(int)

print(f"Churned     : {churn_df['churned'].sum():,} ({churn_df['churned'].mean()*100:.1f}%)")
print(f"Not churned : {(churn_df['churned']==0).sum():,}")

# ── Feature engineering ──────────────────────────────────────
features = ["recency_days", "frequency", "monetary", 
            "r_score", "f_score", "m_score"]

X = churn_df[features].fillna(0)
y = churn_df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")

# ── Train model ──────────────────────────────────────────────
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("\n── Classification report ──────────────────────────────")
print(classification_report(y_test, y_pred, target_names=["Active", "Churned"]))
print(f"ROC-AUC score: {roc_auc_score(y_test, y_proba):.4f}")

4. Churn visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Churn prediction — Random Forest", fontsize=14, fontweight="bold")

# 1. Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Active", "Churned"],
            yticklabels=["Active", "Churned"])
axes[0].set_title("Confusion matrix")
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# 2. ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.3f}")
axes[1].plot([0,1], [0,1], "k--", linewidth=1)
axes[1].set_title("ROC curve")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].legend()

# 3. Feature importance
importances = pd.Series(clf.feature_importances_, index=features).sort_values()
importances.plot(kind="barh", ax=axes[2], color=sns.color_palette("husl", 6))
axes[2].set_title("Feature importance")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("notebooks/churn_model.png", dpi=150, bbox_inches="tight")
plt.show()

# Add churn probability back to customer data
churn_df["churn_probability"] = clf.predict_proba(X)[:, 1]
print("\nTop 10 customers most likely to churn:")
print(churn_df.nlargest(10, "churn_probability")[
    ["customer_id", "recency_days", "frequency", "monetary", "churn_probability"]
].to_string(index=False))

5. Churn Segmentation

In [ ]:
# ── Prepare features ─────────────────────────────────────────
seg_features = ["recency_days", "frequency", "monetary"]
seg_df = rfm[seg_features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(seg_df)

# ── Find optimal k with elbow method ─────────────────────────
inertias    = []
silhouettes = []
k_range     = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("K-Means — finding optimal clusters", fontsize=13, fontweight="bold")

axes[0].plot(k_range, inertias, marker="o", linewidth=2)
axes[0].set_title("Elbow method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_range, silhouettes, marker="o", linewidth=2, color="green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Score (higher = better)")

plt.tight_layout()
plt.show()

# ── Train final model with k=4 ───────────────────────────────
OPTIMAL_K = 4
km_final  = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm["cluster"] = km_final.fit_predict(X_scaled)

# Label clusters by average monetary value
cluster_summary = rfm.groupby("cluster").agg(
    customers   = ("customer_id", "count"),
    avg_recency = ("recency_days", "mean"),
    avg_frequency = ("frequency", "mean"),
    avg_spend   = ("monetary", "mean")
).round(1).sort_values("avg_spend", ascending=False)

cluster_summary["label"] = ["High Value", "Mid Value", "Low Value", "Dormant"]
print("\nCluster summary:")
print(cluster_summary.to_string())

6. Segmentation visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Customer segmentation — K-Means (k=4)", fontsize=14, fontweight="bold")

colors = sns.color_palette("husl", OPTIMAL_K)

# 1. Recency vs Monetary scatter
for i, (cluster_id, label) in enumerate(zip(
        cluster_summary.index, cluster_summary["label"])):
    mask = rfm["cluster"] == cluster_id
    axes[0].scatter(rfm[mask]["recency_days"], rfm[mask]["monetary"],
                    label=label, alpha=0.6, s=40, color=colors[i])

axes[0].set_title("Recency vs monetary by segment")
axes[0].set_xlabel("Recency (days since last order)")
axes[0].set_ylabel("Total spend ($)")
axes[0].legend()

# 2. Cluster size and average spend
cluster_labels = cluster_summary["label"].tolist()
bar_x = range(len(cluster_labels))
bars  = axes[1].bar(bar_x, cluster_summary["avg_spend"], color=colors)
axes[1].set_xticks(bar_x)
axes[1].set_xticklabels(cluster_labels)
axes[1].set_title("Average spend per segment")
axes[1].set_ylabel("Avg spend ($)")

for bar, count in zip(bars, cluster_summary["customers"]):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 50,
                 f"n={count}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("notebooks/segmentation.png", dpi=150, bbox_inches="tight")
plt.show()

7. Sales forecasting

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Build monthly revenue time series ────────────────────────
ts = orders[orders["status"] == "completed"].copy()
ts["order_date"] = pd.to_datetime(ts["order_date"])
ts["period"] = ts["order_date"].dt.to_period("M")

monthly_rev = ts.groupby("period")["order_total"].sum().reset_index()
monthly_rev["period_idx"] = range(len(monthly_rev))
monthly_rev["period_str"] = monthly_rev["period"].astype(str)

# Add time features
monthly_rev["month_num"] = monthly_rev["period"].dt.month
monthly_rev["sin_month"] = np.sin(2 * np.pi * monthly_rev["month_num"] / 12)
monthly_rev["cos_month"] = np.cos(2 * np.pi * monthly_rev["month_num"] / 12)

# ── Train/test split — last 3 months as test ─────────────────
train = monthly_rev[:-3]
test  = monthly_rev[-3:]

feat_cols = ["period_idx", "sin_month", "cos_month"]

model = LinearRegression()
model.fit(train[feat_cols], train["order_total"])

train_pred = model.predict(train[feat_cols])
test_pred  = model.predict(test[feat_cols])

mae  = mean_absolute_error(test["order_total"], test_pred)
rmse = mean_squared_error(test["order_total"], test_pred) ** 0.5

print(f"Forecast MAE  : ${mae:,.2f}")
print(f"Forecast RMSE : ${rmse:,.2f}")

# ── Forecast next 3 months ───────────────────────────────────
last_idx   = monthly_rev["period_idx"].max()
last_period = monthly_rev["period"].max()

future_rows = []
for i in range(1, 4):
    future_period = last_period + i
    future_rows.append({
        "period_idx": last_idx + i,
        "month_num":  future_period.month,
        "sin_month":  np.sin(2 * np.pi * future_period.month / 12),
        "cos_month":  np.cos(2 * np.pi * future_period.month / 12),
        "period_str": str(future_period),
    })

future_df   = pd.DataFrame(future_rows)
future_pred = model.predict(future_df[feat_cols])

print("\nNext 3 month revenue forecast:")
for period, pred in zip(future_df["period_str"], future_pred):
    print(f"  {period} : ${pred:>10,.2f}")

8. Forecast visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle("ShopStream — Sales forecast", fontsize=14, fontweight="bold")

# Actual
ax.plot(monthly_rev["period_str"], monthly_rev["order_total"],
        label="Actual revenue", marker="o", linewidth=2, color="#2196F3")

# Fitted line on train
ax.plot(train["period_str"], train_pred,
        label="Model fit (train)", linewidth=1.5,
        linestyle="--", color="#4CAF50")

# Test predictions
ax.plot(test["period_str"], test_pred,
        label="Model fit (test)", linewidth=1.5,
        linestyle="--", color="#FF9800")

# Future forecast
forecast_x = list(test["period_str"].iloc[-1:]) + future_df["period_str"].tolist()
forecast_y = list(test_pred[-1:]) + list(future_pred)
ax.plot(forecast_x, forecast_y,
        label="Forecast (next 3 months)", linewidth=2,
        linestyle=":", marker="^", color="#E91E63", markersize=8)

ax.axvline(x=test["period_str"].iloc[0], color="gray",
           linestyle="--", alpha=0.5, label="Train/test split")

ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.legend()
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("notebooks/sales_forecast.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nModel MAE  : ${mae:,.2f}")
print(f"Model RMSE : ${rmse:,.2f}")
print("\nPhase 5 complete!")